In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime,UTC
import uuid

In [0]:
%sql
USE CATALOG novadb_lakehouse;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
gold_run_id = str(uuid.uuid4())

run_ts_str = datetime.now(UTC).strftime("%Y-%m-%d %H-%M-%S")
run_dt_str = datetime.now(UTC).strftime("%Y-%m-%d")

print(f"run_ts_str = {run_ts_str}")
print(f"run_dt_str = {run_dt_str}")

In [0]:
%sql

CREATE TABLE IF NOT EXISTS novadb_lakehouse.gold.Ingestion_control(
    Layer STRING,
    Tblname STRING,
    last_successful_silver_runid STRING,
    last_successful_silver_run_ts TIMESTAMP,
    rows_merged BIGINT,
    run_status STRING,
    gold_run_id STRING,
    Updated_at TIMESTAMP
) USING DELTA

In [0]:
def upsert_to_gold(df_source,target_table,join_key):

    if spark.catalog.tableExists(target_table):
        
        dt = DeltaTable.forName(spark,target_table)
        dt.alias("target").merge(
            df_source.alias("source"),
            "target."+ join_key +" = source."+ join_key
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    else:

        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(tablename:str):

    ctrl = ( 
                spark.read.table("novadb_lakehouse.gold.ingestion_control") \
                .filter(
                    (col("Layer") == "gold") &
                    (col("Tblname") == tablename) &
                    (col("run_status") == "success")
                )
                .orderBy(col("updated_at").desc())
                .limit(1)
            )

    rows = ctrl.collect()

    if not rows:
        return None
    
    return rows[0]['last_successful_silver_run_ts']

In [0]:
def upsert_gold_control(tblname,last_successful_silver_runid, last_successful_silver_run_ts,rows_merged,gold_run_id):
    
    ctrl_df = spark.createDataFrame(
        [
            (
                "gold",
                tblname,
                last_successful_silver_runid,
                last_successful_silver_run_ts,
                int(rows_merged),
                "success",
                gold_run_id,
                datetime.now(UTC)
            )
        ],

        schema= """
        Layer STRING,
        Tblname STRING,
        last_successful_silver_runid STRING,
        last_successful_silver_run_ts TIMESTAMP,
        rows_merged BIGINT,
        run_status STRING,
        gold_run_id STRING,
        Updated_at TIMESTAMP
        """
    )

    target_control_df = DeltaTable.forName(spark, "novadb_lakehouse.gold.ingestion_control")

    target_control_df.alias("target").merge(
        ctrl_df.alias("source"),
        "target.tblname = source.tblname"
    ) \
    .whenMatchedUpdate(set = {
        "target.last_successful_silver_runid" : "source.last_successful_silver_runid",
        "target.last_successful_silver_run_ts" : "source.last_successful_silver_run_ts",
        "target.rows_merged" : "source.rows_merged",
        "target.run_status" : "source.run_status",
        "target.gold_run_id" : "source.gold_run_id",
        "target.updated_at" : "source.updated_at"
    }) \
    .whenNotMatchedInsertAll() \
    .execute()



In [0]:
last_gold_ts = get_last_processed_silver_ts("order_information")

print("Last Processed Silver Processed Timestamp for Gold = ", last_gold_ts)

silver_orders_current = spark.read.table("novadb_lakehouse.silver.orders_good")
silver_payments_current = spark.read.table("novadb_lakehouse.silver.payments_good")
silver_products_current = spark.read.table("novadb_lakehouse.silver.products_good")

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_payments = silver_payments_current
    changed_products = silver_products_current
else:
    changed_orders = silver_orders_current.filter(col("updated_at") > last_gold_ts)
    changed_payments = silver_payments_current.filter(col("processed_at") > last_gold_ts)
    changed_products = silver_products_current.filter(col("updated_at") > last_gold_ts)

changed_orders_count = changed_orders.count()
changed_payments_count = changed_payments.count()
changed_products_count = changed_products.count()

print("changed_orders_count = ", changed_orders_count)
print("changed_payments_count = ", changed_payments_count)
print("changed_products_count = ", changed_products_count)



In [0]:
impacted_from_orders = changed_orders.select(col("order_id")).distinct() 
impacted_from_payments = changed_payments.select(col("order_id")).distinct()
impacted_from_products = changed_products.alias("p")\
    .join(
        silver_orders_current.alias("o") ,
        col("p.product_id") == col("o.product_id"),
        "inner"
        )\
        .select(col("o.order_id")).distinct()

impacted_order_id = impacted_from_orders.union(impacted_from_payments).union(impacted_from_products).distinct()

print("impacted_order_id.count() = ", impacted_order_id.count())
display(impacted_order_id.orderBy("order_id"))             



In [0]:
impacted_orders = silver_orders_current.alias("oc").join(
    impacted_order_id.alias("ic"),
    col("oc.order_id") == col("ic.order_id"),
    "inner"
    ).drop(col("ic.order_id"))

gold_delta = ( impacted_orders.alias("o")
.join(silver_products_current.alias("p"), col("o.product_id") == col("p.product_id"), "inner")
.join(silver_payments_current.alias("pay"), col("o.order_id") == col("pay.order_id"), "inner")
)\
.select(
    col("o.order_id"),
    col("o.customer_id"),
    col("p.product_id"),
    col("p.product_name"),
    col("p.category"),
    col("p.price").alias("product_price"),
    col("o.order_status"),
    col("o.order_amount"),
    col("pay.payment_id"),
    col("pay.payment_status"),
    col("pay.paid_amount"),
    col("o.order_date"),
    col("o.order_year"),
    col("o.order_month"),
    col("o.order_day"),
    greatest(col("o.updated_at"), col("p.updated_at"), col("pay.processed_at")).alias("gold_updated_ts")
)\
.dropDuplicates(["order_id"])\
.withColumn("payment_completion_ratio",
            when(col("o.order_amount") > 0, col("pay.paid_amount")/col("o.order_amount"))
            .otherwise(lit(0.0))
            )\
.withColumn("payment_state",
            when(col("order_amount") == 0, "Invalid amount number")
            .when(col("payment_completion_ratio") == 0, "unpaid")
            .when(col("payment_completion_ratio") == 1,"paid")
            .when(col("payment_completion_ratio") < 1,"Partially Paid")
            .when(col("payment_completion_ratio") > 1,"Overpaid")
            .otherwise(lit("Unknown"))
            )\
.withColumn("gold_update_date", col("gold_updated_ts"))

print("gold_delta_rows = ", gold_delta.count())
display(gold_delta)

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta,"novadb_lakehouse.gold.order_information","order_id")
else:
    print("No new data to be ingested")

In [0]:
%sql
SELECT * FROM novadb_lakehouse.gold.order_information;

In [0]:
if not spark.catalog.tableExists("novadb_lakehouse.gold.order_information_scd2"):
    spark.sql(
"""
        create table novadb_lakehouse.gold.order_information_scd2
        using delta
        as 
        select * , 
        cast(null as timestamp) as valid_from_ts, 
        cast(null as timestamp) as valid_to_ts,
        true as is_current
        from novadb_lakehouse.gold.order_information
        where 1 = 0
        """
        )
    
if gold_delta.count() > 0:
    
    gold_delta.createOrReplaceTempView("gold_delta_view")

    spark.sql(
        """
        MERGE INTO novadb_lakehouse.gold.order_information_scd2 t
        USING gold_delta_view s
        ON t.order_id = s.order_id AND t.is_current = true

        WHEN MATCHED AND (
            NOT (t.order_status <=> s.order_status) OR
            NOT (t.order_amount <=> s.order_amount) OR
            NOT (t.paid_amount <=> s.paid_amount) OR
            NOT (t.payment_id <=> s.payment_id) OR
            NOT (t.category <=> s.category) OR
            NOT (t.product_name <=> s.product_name) OR
            NOT (t.product_price <=> s.product_price)
        )
        THEN UPDATE SET
            t.valid_to_ts = s.gold_updated_ts,
            t.is_current = false
        """
    )

    spark.sql(
        """
        insert into novadb_lakehouse.gold.order_information_scd2
        select s.* , 
        cast(s.gold_updated_ts as timestamp) as valid_from_ts, 
        cast(null as timestamp) as valid_to_ts,
        true as is_current
        from gold_delta_view s
        left join novadb_lakehouse.gold.order_information_scd2 t
        on s.order_id = t.order_id and t.is_current = true
        where t.order_id is NULL or (
            NOT (t.order_status <=> s.order_status) OR
            NOT (t.order_amount <=> s.order_amount) OR
            NOT (t.paid_amount <=> s.paid_amount) OR
            NOT (t.payment_id <=> s.payment_id) OR
            NOT (t.category <=> s.category) OR
            NOT (t.product_name <=> s.product_name) OR
            NOT (t.product_price <=> s.product_price)
        )
        """
    )


In [0]:
if gold_delta.count() > 0:
    
    impacted_category = gold_delta.select("category").filter(col("category").isNotNull()).distinct()


    category_performance_delta = (spark.read.table("novadb_lakehouse.gold.order_information").join(impacted_category, "category", "inner").groupBy("category").agg(
        countDistinct("order_id").alias("total_orders"),
        sum(
            when(col("order_amount") > 0 , col("order_amount")).otherwise(lit(0.0))
        ).alias("total_revenue"),
        sum(
            when(col("paid_amount") > 0 , col("paid_amount")).otherwise(lit(0.0))
        ).alias("total_paid"),
        avg(col("payment_completion_ratio")).alias("average_payment_completion_ratio"),
        (
            sum(
                when(col("payment_status") == "failed",1).otherwise(0)) /count("*").cast("double")
        ).alias("payment_failure_rate")
    )
    )

    upsert_to_gold(category_performance_delta,"novadb_lakehouse.gold.category_performance","category")
    
        

In [0]:
%sql

SELECT * FROM novadb_lakehouse.gold.category_performance;

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS novadb_lakehouse.gold.gold_snapshots_vol;

In [0]:
latest_order_path = ("/Volumes/novadb_lakehouse/gold/gold_snapshots_vol/gold_latest/order_information")
latest_category_path = ("/Volumes/novadb_lakehouse/gold/gold_snapshots_vol/gold_latest/category_performance")

historical_order_path = (f"/Volumes/novadb_lakehouse/gold/gold_snapshots_vol/gold_snapshots/order_information/run_date={run_dt_str}/run_ts={run_ts_str}")
historical_category_path = (f"/Volumes/novadb_lakehouse/gold/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_dt_str}/run_ts={run_ts_str}")

spark.read.table("novadb_lakehouse.gold.order_information").write.mode("overwrite").format("parquet").save(latest_order_path)

spark.read.table("novadb_lakehouse.gold.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("novadb_lakehouse.gold.order_information").write.mode("overwrite").format("parquet").save(historical_order_path)

spark.read.table("novadb_lakehouse.gold.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print("latest_order_path : ",latest_order_path)
print("latest_category_path : ",latest_category_path)
print("historical_order_path : ",historical_order_path)
print("historical_category_path : ",historical_category_path)



In [0]:
display(silver_orders_current)

In [0]:
latest_silver_ts = silver_orders_current.agg(max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

latest_silver_run_id = ( silver_orders_current.filter(col("bronze_ingested_at") == latest_silver_ts).agg(max("silver_run_id").alias("mx")).collect()[0]["mx"]) if latest_silver_ts is not None else None

print(gold_delta.count())

upsert_gold_control("order_information",latest_silver_run_id,latest_silver_ts,gold_delta.count(),gold_run_id)

display(spark.table("novadb_lakehouse.gold.ingestion_control"))
